# P2W1D2: Prompt Technique and Optimization

Isi dan tujuannya sama persis dengan `README.md` (structured prompting, chain-of-thought, few-shot, evaluasi & optimasi). Baca dulu `README.md` (terutama "Konsep Guided Hands-on" dan "Hands-On-Lab") sebelum mengerjakan notebook ini.

**Pilih salah satu provider** -- cukup salah satu, tidak perlu keduanya:

- **Claude (Anthropic API)** -- butuh API key dari [console.anthropic.com](https://console.anthropic.com/).
- **ChatGPT (OpenAI API)** -- butuh API key dari [platform.openai.com](https://platform.openai.com/).

Salin `.env.example` menjadi `.env`, lalu tentukan `PROVIDER` dan isi API key provider yang dipilih di file `.env`.

**Biaya:** tiap kali sebuah sel memanggil `ask_llm(...)`, itu satu pemanggilan API berbayar. Model default sudah dipilihkan yang murah & cepat (Claude Haiku `claude-haiku-4-5` atau OpenAI `gpt-5.6-luna`) -- estimasi total biaya untuk seluruh notebook di bawah USD 1, untuk provider mana pun.


## 1. Install Package

In [1]:
# anthropic: SDK resmi untuk Claude API
# openai: SDK resmi untuk OpenAI API
# python-dotenv: membaca provider dan API key dari file .env

%pip install -q anthropic openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Kredensial lewat file `.env`

Buat file `.env` di folder yang sama dengan notebook ini (lihat `.env.example`), lalu isi
kredensial milikmu sendiri. Kode hanya **membaca** nilainya — key tidak pernah ditulis di notebook.

Isi minimal `.env`:

```
PROVIDER=claude            # pilih: claude atau openai
ANTHROPIC_API_KEY=...      # kalau PROVIDER = "claude"
OPENAI_API_KEY=...         # kalau PROVIDER = "openai"
```

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)

# Atur PROVIDER dan API key di file .env. Cukup pilih salah satu provider.
PROVIDER = os.getenv("PROVIDER", "claude").strip().lower()

if PROVIDER == "claude":
    import anthropic

    api_key = os.getenv("ANTHROPIC_API_KEY", "").strip()
    if not api_key:
        raise ValueError("ANTHROPIC_API_KEY belum diisi di file .env.")

    client = anthropic.Anthropic(api_key=api_key)
    MODEL_NAME = "claude-haiku-4-5"  # Claude Haiku: murah & cepat, cukup untuk latihan prompting ini

    def ask_llm(prompt: str) -> str:
        """Kirim satu prompt ke Claude, kembalikan teks jawabannya."""
        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=500,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text

elif PROVIDER == "openai":
    import openai

    api_key = os.getenv("OPENAI_API_KEY", "").strip()
    if not api_key:
        raise ValueError("OPENAI_API_KEY belum diisi di file .env.")

    client = openai.OpenAI(api_key=api_key)
    MODEL_NAME = "gpt-5.6-luna"  # model murah & cepat dari OpenAI, cukup untuk latihan prompting ini

    def ask_llm(prompt: str) -> str:
        """Kirim satu prompt ke ChatGPT/OpenAI, kembalikan teks jawabannya."""
        response = client.chat.completions.create(
            model=MODEL_NAME,
            max_completion_tokens=500,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content

else:
    raise ValueError('PROVIDER harus "claude" atau "openai"')

print(f"Setup selesai. Provider aktif: {PROVIDER} ({MODEL_NAME})")

Setup selesai. Provider aktif: openai (gpt-5.6-luna)


In [4]:
# Enam tiket pelanggan Northstar Market, identik dengan tabel di README.md (Studi Kasus).
TICKETS = {
    "T1": "Paket saya seharusnya sampai 3 hari lalu tapi status masih 'dalam pengiriman'. "
          "Saya sudah coba hubungi kurir tapi tidak ada respons. Tolong bantu cek.",
    "T2": "Blender yang saya beli minggu lalu retak di bagian wadah kaca padahal baru dipakai dua kali.",
    "T3": "Saya sudah mengajukan refund sejak tanggal 2, tapi sampai sekarang dana belum masuk ke "
          "rekening. Ini sudah 12 hari kerja.",
    "T4": "Apakah promo gratis ongkir masih berlaku untuk pembelian di atas Rp150.000 minggu ini?",
    "T5": "Wah, hebat sekali, paket saya dikirim ke alamat yang salah dan sekarang saya harus "
          "repot-repot mengurusnya sendiri. Terima kasih banyak, Northstar Market.",
    "T6": "Kualitas kemasan rapi, barang sampai lebih cepat dari estimasi, dan sesuai deskripsi. "
          "Akan belanja lagi di sini!",
}

# Skema output yang diminta (sama seperti README.md):
# {"kategori": "pengiriman | produk_cacat | pembayaran_refund | pertanyaan_umum | pujian",
#  "sentimen": "positif | netral | negatif",
#  "prioritas": "rendah | sedang | tinggi",
#  "ringkasan": "ringkasan inti pesan, maksimal 2 kalimat",
#  "tindakan_disarankan": "satu tindakan konkret untuk agent"}

## 3. Structured Prompting

Lengkapi `STRUCTURED_PROMPT` dengan 4 komponen pada **Konsep 1** (Context & Persona, Instruction/Task, Data Input, Output Format). Gunakan token `<<PESAN>>` di dalam `<data>...</data>` -- token ini akan diganti otomatis dengan `.replace(...)` oleh isi pesan pelanggan sebelum dikirim ke API. Uji dengan tiket **T1**.

In [6]:
STRUCTURED_PROMPT = """
[ROLE]
tentukan sentimen, prioritas, ringkasan dan tindakan_disarankan berdasarkan

[CONTEXT]
data object tiket memiliki beberapa transaksi yang diidentiikasi dengan key inkremental seperti T1, T2, T2, dan seterusnya kemudian value berupa kalimat ulasan.

[TASK]
 baca satu pesan suatu tiket, lalu tentukan kategori, sentimen, prioritas, ringkasan, kemudian masukkan dalam hasil dalam format dua json(untuk alasan dan hasil) .
<data>
<<PESAN>>
</data>

[FORMAT]
{"kategori" : alasan,
"sentimen" : alasan,
"prioritas" : alasan
}
}
{"kategori": "pengiriman | produk_cacat | pembayaran_refund | pertanyaan_umum | pujian",
 "sentimen": "positif | netral | negatif",
 "prioritas": "rendah | sedang | tinggi",
 "ringkasan": "ringkasan inti pesan, maksimal 2 kalimat",
 "tindakan_disarankan": "satu tindakan konkret untuk agent"}
"""

prompt = STRUCTURED_PROMPT.replace("<<PESAN>>", TICKETS["T1"])
print(ask_llm(prompt))

{"kategori":"Pengiriman terlambat dan status paket masih dalam pengiriman.","sentimen":"Negatif karena paket terlambat tiga hari dan kurir tidak merespons.","prioritas":"Tinggi karena keterlambatan sudah melewati estimasi dan pelanggan meminta bantuan pengecekan."}
{"kategori":"pengiriman","sentimen":"negatif","prioritas":"tinggi","ringkasan":"Paket terlambat tiga hari dan statusnya masih 'dalam pengiriman'. Pelanggan sudah mencoba menghubungi kurir tetapi tidak mendapat respons.","tindakan_disarankan":"Segera lacak status paket dan hubungi pihak kurir untuk memberikan pembaruan kepada pelanggan."}


## 4. Chain-of-Thought

Salin isi `STRUCTURED_PROMPT` ke `REASONING_PROMPT` (`[ROLE]` dan `[CONTEXT]` cukup copy-paste persis, tidak perlu ditulis ulang -- fokus Anda ada di `[TASK]` dan `[FORMAT]`), lalu tambahkan instruksi penalaran bertahap (boleh zero-shot CoT dengan salah satu frasa pemicu pada **Konsep 2**, atau manual CoT dengan contoh reasoning) sebelum model menuliskan JSON akhir. Tentukan urutan output: baris "Penalaran:", baris "===", baru JSON. Uji khusus pada tiket **T5**.

In [ ]:
REASONING_PROMPT = """
[ROLE]
tentukan sentimen, prioritas, ringkasan dan tindakan_disarankan berdasarkan

[CONTEXT]
data object tiket memiliki beberapa transaksi yang diidentiikasi dengan key inkremental seperti T1, T2, T2, dan seterusnya kemudian value berupa kalimat ulasan.

[TASK]
 baca satu pesan suatu tiket, lalu tentukan kategori, sentimen, prioritas, ringkasan, kemudian masukkan dalam hasil dalam format.
<data>
<<PESAN>>
</data>

[FORMAT]
 tentukan urutan jawaban: baris "Penalaran:" (maksimal 3 kalimat), baris "===", lalu satu objek
JSON dengan field kategori, sentimen, prioritas, ringkasan, tindakan_disarankan, tanpa teks tambahan
setelahnya. Sertakan kembali daftar nilai valid untuk tiap field (sama seperti pada STRUCTURED_PROMPT),
jangan hanya menyebut nama field-nya saja -- ini penting supaya nilainya tetap konsisten antar run.

"""

prompt = REASONING_PROMPT.replace("<<PESAN>>", TICKETS["T5"])
print(ask_llm(prompt))

**Catatan penalaran:** Apakah bagian penalaran model menangkap nada sarkastik pada T5? Jelaskan singkat di sel markdown ini.

## 5. Few-shot Prompting

Salin isi `REASONING_PROMPT` ke `FEWSHOT_PROMPT` (`[ROLE]` dan `[CONTEXT]` tetap copy-paste, tidak perlu ditulis ulang). Notebook ini sudah menyediakan satu contoh sarkasme di bagian `[CONTOH]` sebagai acuan format mengikuti anatomi pada **Konsep 3** -- tugas Anda menambahkan **minimal 1 contoh lagi** dengan kategori yang jelas berbeda (disarankan kategori yang jelas positif seperti `pujian`), supaya model tidak hanya meniru pola kategori negatif/netral yang sudah ada. Uji pada tiket **T5** dan **T6**.

In [ ]:
FEWSHOT_PROMPT = """
[ROLE]
(Copy-paste persis ROLE dari REASONING_PROMPT di atas -- tidak perlu ditulis ulang.)

[CONTEXT]
(Copy-paste persis CONTEXT dari REASONING_PROMPT di atas -- tidak perlu ditulis ulang.)

[CONTOH]
Pesan: "Wow, keren banget, barang pesanan saya dikirim ke kota yang salah dan saya harus mengurusnya sendiri. Terima kasih banyak ya."
Penalaran: Nada sindiran ("wow, keren banget", "terima kasih banyak") menandakan kekecewaan atas salah kirim alamat, bukan pujian tulus.
===
{"kategori": "pengiriman", "sentimen": "negatif", "prioritas": "tinggi", "ringkasan": "Barang dikirim ke alamat/kota yang salah, pelanggan kecewa dan harus mengurus sendiri.", "tindakan_disarankan": "Lacak paket dan koordinasikan pengiriman ulang."}

TODO: tambahkan MINIMAL 1 contoh lagi dengan kategori yang jelas berbeda dari contoh di atas (disarankan
kategori yang jelas positif, mis. pujian), supaya model tidak hanya belajar dari kasus negatif/netral.
Ikuti format yang sama: "Pesan: ... / Penalaran: ... / === / {json satu baris}".

[TASK]
TODO: salin/perbaiki task dari REASONING_PROMPT, minta model mengikuti pola contoh di atas untuk pesan baru.
<data>
<<PESAN>>
</data>

[FORMAT]
TODO: sama seperti REASONING_PROMPT (baris "Penalaran:", baris "===", lalu JSON dengan daftar nilai
valid tiap field, jangan hanya nama field-nya saja).
"""

for ticket_id in ["T5", "T6"]:
    prompt = FEWSHOT_PROMPT.replace("<<PESAN>>", TICKETS[ticket_id])
    print(f"===== {ticket_id} =====")
    print(ask_llm(prompt))
    print()

## 6. Evaluasi & Optimasi hingga Stabil

`FINAL_PROMPT` di kode starter sudah otomatis mengambil isi `FEWSHOT_PROMPT` (`FINAL_PROMPT = FEWSHOT_PROMPT`), jadi tidak perlu menulis ulang dari nol di titik awal. Uji pada **minimal 3 tiket berbeda**, dan untuk tiap tiket jalankan selnya **2 kali berturut-turut** (tanpa mengubah kode) untuk mengecek Konsistensi (**Konsep 4**). Kalau ada dimensi yang belum baik, revisi `FINAL_PROMPT` lalu jalankan ulang, mengikuti siklus Draft-Uji-Evaluasi-Refine pada **Konsep 5**, sampai memenuhi kriteria "Tanda Prompt Sudah Stabil".

> **Opsional:** pada tahap Refine, Anda boleh mencoba **Self-Check Prompting** (README Konsep 6) -- minta model mereview dan mengidentifikasi kelemahan dari keluarannya sendiri sebelum Anda merevisi `FINAL_PROMPT` secara manual. Teknik ini tidak wajib dan tidak memengaruhi penilaian.

In [ ]:
FINAL_PROMPT = FEWSHOT_PROMPT  # Otomatis sama dengan FEWSHOT_PROMPT di titik awal, tidak perlu ditulis ulang.
# TODO: revisi teksnya di atas (bukan tulis ulang dari nol) kalau ada dimensi evaluasi yang belum baik.

TIKET_UJI = ["T1", "T3", "T5"]  # TODO: boleh ganti/tambah, minimal 3 tiket berbeda

for ticket_id in TIKET_UJI:
    prompt = FINAL_PROMPT.replace("<<PESAN>>", TICKETS[ticket_id])
    print(f"===== {ticket_id} =====")
    print(ask_llm(prompt))
    print()

# Jalankan ulang sel ini beberapa kali (tanpa mengubah kode) untuk tiket yang sama,
# lalu bandingkan hasilnya secara manual -- itulah cara mengecek Konsistensi.

### Evaluasi 5 Dimensi Kualitas Output

Gunakan tabel Dimensi Evaluasi pada **Konsep 4** di README sebagai acuan, isi berdasarkan hasil run di atas.

| Dimensi | Catatan Hasil Uji | Sudah Baik? (Ya/Belum) |
| --- | --- | --- |
| Konsistensi | | |
| Kesesuaian Format | | |
| Akurasi | | |
| Relevansi | | |
| Panjang Jawaban | | |

### Log Iterasi (Draft - Uji - Evaluasi - Refine)

| Percobaan | Perubahan yang Dilakukan pada `FINAL_PROMPT` | Hasil Uji Singkat | Catatan |
| --- | --- | --- | --- |
| 1 | | | |
| 2 | | | |
| 3 | | | |

**Kesimpulan stabil:** Pada percobaan keberapa `FINAL_PROMPT` Anda dianggap memenuhi "Tanda Prompt Sudah Stabil" (lihat Konsep 5 di README), dan perubahan apa yang paling berpengaruh?

---

## Ringkasan

| | Jumlah Iterasi | Status Akhir |
| --- | --- | --- |
| Hasil | | Stabil / Belum Stabil |
